In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import joblib

In [2]:
df = pd.read_csv("customer_subscription_churn_usage_patterns.csv")

In [4]:
df.head(3)

,user_id,signup_date,plan_type,monthly_fee,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago,churn
0,1,2023-04-15,Premium,699,1.1,4,1,8,14,Yes
1,2,2023-08-27,Premium,699,2.6,6,0,35,1,Yes
2,3,2023-10-12,Premium,699,14.3,8,3,2,14,Yes


In [5]:
# Drop rows with any nulls in our feature columns
FEATURES = [
    "avg_weekly_usage_hours",
    "support_tickets",
    "payment_failures",
    "tenure_months",
    "last_login_days_ago",
]
 
df = df.dropna(subset=FEATURES).copy()

In [6]:
# Clip sensible bounds (same logic as before, driven by real data ranges)
df["avg_weekly_usage_hours"] = df["avg_weekly_usage_hours"].clip(0, 40)
df["tenure_months"]          = df["tenure_months"].clip(0, 120)
df["last_login_days_ago"]    = df["last_login_days_ago"].clip(0, 180)
df["support_tickets"]        = df["support_tickets"].clip(0, 50)
df["payment_failures"]       = df["payment_failures"].clip(0, 20)

In [7]:
df[FEATURES].describe().round(2)

,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago
count,2800.00,2800.00,2800.00,2800.00,2800.00
mean,12.89,3.89,2.49,18.61,30.00
std,7.11,2.61,1.69,10.37,17.85
min,0.50,0.00,0.00,1.00,0.00
25%,6.70,2.00,1.00,10.00,14.00
50%,12.80,4.00,2.00,18.00,30.00
75%,19.20,6.00,4.00,27.00,46.00
max,25.00,8.00,5.00,36.00,60.00


In [8]:
# ── 3. Feature matrix ────────────────────────────────────────────────────────
X = df[FEATURES].values   # shape (n_samples, 5)
 


In [9]:
# ── 4. Standardise ───────────────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [10]:
gmm = GaussianMixture(
    n_components=3,
    covariance_type="full",
    random_state=42,
    n_init=10,
    max_iter=300,
)
gmm.fit(X_scaled)

,n_components,3
,covariance_type,'full'
,tol,0.001
,reg_covar,1e-06
,max_iter,300
,n_init,10
,init_params,'kmeans'
,weights_init,None
,means_init,None
,precisions_init,None
,random_state,42


In [22]:
# ── 6. Label clusters by avg_weekly_usage_hours ──────────────────────────────
labels     = gmm.predict(X_scaled)
df         = df.copy()
df["cluster"] = labels
 

In [32]:
#Initial Mapping
# Lowest usage  → At-Risk
# Middle usage  → Standard
# Highest usage → Power
label_map = {
    int(cluster_usage.index[0]): "At-Risk/Churning Users",
    int(cluster_usage.index[1]): "Standard Users",
    int(cluster_usage.index[2]): "Power Users",
}
print("Label map:", label_map)
print("Cluster sizes:", df["cluster"].value_counts().to_dict())

Label map: {1: 'At-Risk/Churning Users', 2: 'Standard Users', 0: 'Power Users'}
Cluster sizes: {0: 992, 2: 958, 1: 850}


In [33]:
# ── 7. Sanity-check: how does churn align with segments? ─────────────────────
df["segment"] = df["cluster"].map(label_map)
print("\nChurn rate by segment:")
print(
    df.groupby("segment")["churn"]
    .apply(lambda x: (x == "Yes").mean())
    .round(3)
)


Churn rate by segment:
segment
At-Risk/Churning Users    0.453
Power Users               0.584
Standard Users            0.669
Name: churn, dtype: float64


In [34]:

# Logical Mapping
label_map = {
    df.groupby("cluster")["payment_failures"].mean().idxmax(): "At-Risk/Churning Users",
    df.groupby("cluster")["avg_weekly_usage_hours"].mean().idxmax(): "Power Users",
}
# Fill the remaining key as 'Standard'
for c in [0, 1, 2]:
    if c not in label_map:
        label_map[c] = "Standard Users"
        
print("Label map:", label_map)
print("Cluster sizes:", df["cluster"].value_counts().to_dict())
 

Label map: {np.int64(2): 'At-Risk/Churning Users', np.int64(0): 'Power Users', 1: 'Standard Users'}
Cluster sizes: {0: 992, 2: 958, 1: 850}


In [35]:
# ── 7. Sanity-check: how does churn align with segments? ─────────────────────
df["segment"] = df["cluster"].map(label_map)
print("\nChurn rate by segment:")
print(
    df.groupby("segment")["churn"]
    .apply(lambda x: (x == "Yes").mean())
    .round(3)
)


Churn rate by segment:
segment
At-Risk/Churning Users    0.669
Power Users               0.584
Standard Users            0.453
Name: churn, dtype: float64


In [36]:
# ── 8. Spot-check predict_proba on a few real rows ───────────────────────────
print("\n── predict_proba on 4 real rows ──")
for _, row in df.sample(4, random_state=7).iterrows():
    x_raw    = np.array([[row[f] for f in FEATURES]])
    x_scaled = scaler.transform(x_raw)
    proba    = gmm.predict_proba(x_scaled)[0]
    dist     = {label_map[i]: round(float(proba[i]) * 100, 2) for i in range(3)}
    print(f"  user_id={int(row['user_id'])} churn={row['churn']} → {dist}")


── predict_proba on 4 real rows ──
  user_id=1637 churn=No → {'Power Users': 65.17, 'Standard Users': 34.83, 'At-Risk/Churning Users': 0.0}
  user_id=2266 churn=No → {'Power Users': 0.22, 'Standard Users': 0.71, 'At-Risk/Churning Users': 99.07}
  user_id=191 churn=No → {'Power Users': 95.3, 'Standard Users': 4.69, 'At-Risk/Churning Users': 0.0}
  user_id=1419 churn=Yes → {'Power Users': 14.26, 'Standard Users': 0.02, 'At-Risk/Churning Users': 85.72}


In [37]:
# ── 9. Serialise ─────────────────────────────────────────────────────────────
joblib.dump(gmm,    "gmm_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(
    {"label_map": label_map, "features": FEATURES},
    "metadata.pkl",
)

['metadata.pkl']